In [ ]:
def localizar_raiz_projeto():
    # Import local para permitir execução isolada desta célula.
    from pathlib import Path as _Path
    for candidato in [_Path.cwd(), *_Path.cwd().parents]:
        if (candidato / 'Etapa_2' / 'output' / 'referencias_processamento.json').exists():
            return candidato
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto QuestionInsight.')

PROJECT_ROOT = localizar_raiz_projeto()
DETAIL_DIR = PROJECT_ROOT / 'Etapa_5' / 'Analise_Detalhada_Misconceptions_PC3'
OUTPUT_DIR = DETAIL_DIR / 'output'
import ast
import json
import re
import shutil
from pathlib import Path
import pandas as pd

csv_detailed = PROJECT_ROOT / 'Etapa_5' / 'output' / 'misconceptions_detalhado_por_usuario.csv'
references_path = PROJECT_ROOT / 'Etapa_2' / 'output' / 'referencias_processamento.json'
references_output = OUTPUT_DIR / 'referencias_por_misconception.json'
pasta_destino = OUTPUT_DIR
def get_subpasta_nome(misconceptions_detectados):
    if pd.isna(misconceptions_detectados) or str(misconceptions_detectados).strip() == '':
        return 'SEM_MISCONCEPTIONS'
    mcs = [mc.strip() for mc in str(misconceptions_detectados).split(',') if mc.strip()]
    return '_'.join(sorted(mcs))


def carregar_referencias(path):
    references_path = Path(path).resolve()
    data = json.loads(references_path.read_text(encoding='utf-8'))
    source_root = Path(data['source_root'])
    if not source_root.is_absolute():
        source_root = (references_path.parent / source_root).resolve()
    data['_source_root'] = str(source_root)
    return data


TP_DESCRICOES = {
    'TP 01': 'Variáveis e Estrutura Sequencial',
    'TP 02': 'Estrutura Condicional Simples e Composta',
    'TP 03': 'Estrutura Condicional Encadeada',
    'TP 04': 'Estrutura de Repetição por Condição',
    'TP 05': 'Vetores e Strings',
    'TP 06': 'Estrutura de Repetição por Contagem',
}

def classificar_tp(texto):
    numeros = [int(x) for x in re.findall(r'\bTP\s*[-:]?\s*0?(\d+)\b', str(texto), flags=re.I)]
    if not numeros:
        return 'Não classificado', 'Assessment sem grupo TP identificado'
    grupo = f'TP {min(max(numeros), 6):02d}'
    return grupo, TP_DESCRICOES.get(grupo, 'Grupo TP não mapeado')

def _assessment_index(data):
    """Indexa assessments por escopo + ID, evitando colisões entre turmas."""
    indice = {}
    for item in data.get('assessments', []):
        path_value = item.get('path') or item.get('filename') or ''
        path_parts = Path(str(path_value).replace('\\', '/')).parts
        assessment_id = Path(str(item.get('filename') or path_value)).stem
        if assessment_id:
            if 'assessments' in path_parts:
                aidx = path_parts.index('assessments')
                scope = '/'.join(path_parts[:aidx])
            else:
                scope = ''
            indice[(scope, assessment_id)] = item
            # Mantém índice global somente para IDs que não colidem.
            indice.setdefault(('', assessment_id), item)
    return indice

def _assessment_metadata(assessments, source_root, code_path, assessment_id, item):
    """Obtém o assessment pelo mesmo escopo físico do código."""
    rel = Path(code_path).resolve().relative_to(source_root.resolve())
    parts = rel.parts
    scope = '/'.join(parts[:parts.index('users')]) if 'users' in parts else ''
    assessment = assessments.get((scope, assessment_id)) or assessments.get(('', assessment_id)) or {}
    # Se o JSON for antigo/incompleto, tenta o .data correspondente no mesmo escopo.
    if not assessment and scope:
        data_path = source_root.joinpath(*scope.split('/'), 'assessments', f'{assessment_id}.data')
        if data_path.is_file():
            try:
                assessment = {'assessment_title': '', 'group': ''}
                for line in data_path.read_text(encoding='utf-8', errors='replace').splitlines():
                    if ':' not in line:
                        continue
                    key, value = line.split(':', 1)
                    key = key.strip('- ').strip()
                    value = value.strip()
                    if key == 'assessment title': assessment['assessment_title'] = value
                    elif key == 'type': assessment['type'] = value
            except OSError:
                pass
    titulo = assessment.get('assessment_title') or assessment.get('title') or item.get('assessment_title') or ''
    grupo = (assessment.get('tp_group') or assessment.get('grupo_tp') or
             assessment.get('group') or item.get('tp_group') or item.get('grupo_tp') or item.get('group'))
    if grupo:
        grupo = str(grupo)
        descricao = (TP_DESCRICOES.get(grupo) or assessment.get('tp_description') or
                     assessment.get('group_description') or item.get('tp_description') or
                     item.get('group_description') or 'Grupo TP registrado nas referências')
    else:
        grupo, descricao = classificar_tp(titulo)
    return assessment, titulo, grupo, descricao, scope

def extrair_codigo_ultimo_submition(execution_path):
    """Extrai somente o código do último SUBMITION com GRADE e sem ERROR."""
    separador = '*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*'
    try:
        conteudo = Path(execution_path).read_text(encoding='utf-8', errors='replace')
    except (OSError, UnicodeError):
        return None
    blocos = [b for b in conteudo.split(separador)
              if re.search(r'^\s*==\s*SUBMITION\b', b, flags=re.I|re.M)]
    if not blocos:
        return None
    ultimo = blocos[-1]
    if (re.search(r'^\s*-+\s*GRADE\s*:', ultimo, flags=re.I|re.M) is None
            or re.search(r'^\s*-+\s*ERROR\s*:', ultimo, flags=re.I|re.M) is not None):
        return None
    match = re.search(r'^\s*--\s*CODE\s*:\s*\n(.*?)(?=^\s*--\s*(?:EXECUTION TIME|TEST CASE|ERROR|GRADE)\s*:|\Z)', ultimo, flags=re.I|re.M|re.S)
    if match is None:
        return None
    codigo = match.group(1).strip('\n\r')
    return codigo if codigo.strip() else None

def indexar_arquivos_executions(data):
    print('Indexando referências dos execution logs...')
    index = {}
    falhas = []
    source_root = Path(data['_source_root'])
    assessments = _assessment_index(data)
    for record in data.get('users', []):
        usuario_id = str(record['id'])
        for item in record.get('files', {}).get('executions', []):
            path = source_root / item['path']
            if not path.is_file():
                continue
            codigo = extrair_codigo_ultimo_submition(path)
            if codigo is None:
                continue
            try:
                ast.parse(codigo)
            except SyntaxError as exc:
                falhas.append({
                    'path': str(path.resolve()),
                    'id_usuario': usuario_id,
                    'assessment_id': str(item.get('assessment') or path.stem.split('_', 1)[0]),
                    'question_id': str(item.get('question') or path.stem.rsplit('_', 1)[-1]),
                    'stage': 'prevalidacao_analise_detalhada',
                    'error_type': type(exc).__name__,
                    'message': str(exc),
                    'line': exc.lineno,
                    'offset': exc.offset,
                })
                continue
            partes = path.stem.rsplit('_', 1)
            assessment_id = str(item.get('assessment') or partes[0])
            questao_id = str(item.get('question') or (partes[1] if len(partes) == 2 else partes[-1]))
            assessment, titulo, grupo, descricao, scope = _assessment_metadata(
                assessments, source_root, path, assessment_id, item
            )
            index[(usuario_id, questao_id)] = {
                'path': str(path), 'assessment': assessment_id, 'question': questao_id,
                'usuario': usuario_id, 'scope': scope, 'assessment_title': titulo,
                'assessment_type': assessment.get('type', item.get('type', '')),
                'tp_group': grupo, 'tp_description': descricao,
            }
    pasta_destino.mkdir(parents=True, exist_ok=True)
    falhas_path = pasta_destino / 'falhas_analise_detalhada_prevalidacao.json'
    falhas_path.write_text(json.dumps({
        'schema_version': 1,
        'descricao': 'Executions excluídos da análise detalhada por erro sintático no último SUBMITION.',
        'criterio': 'último SUBMITION com GRADE, sem ERROR e código aceito por ast.parse.',
        'total_failures': len(falhas),
        'failures': falhas,
    }, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print(f'   {len(index)} referências de execution logs indexadas.')
    print(f'   Falhas sintáticas excluídas: {len(falhas)}; relatório: {falhas_path.resolve()}')
    return index
def criar_referencias_pc3():
    print(f'Lendo CSV: {csv_detailed}')
    # A saída é regenerada do zero para não misturar grupos de execuções antigas.
    pasta_destino.mkdir(parents=True, exist_ok=True)
    for item_saida in pasta_destino.iterdir():
        if item_saida.is_dir():
            shutil.rmtree(item_saida)
        elif item_saida.name in {'referencias_por_misconception.json', 'analise_detalhada_resultados.json'}:
            item_saida.unlink()
    df = pd.read_csv(csv_detailed)
    data = carregar_referencias(references_path)
    index = indexar_arquivos_executions(data)
    grupos = {}
    encontrados = 0
    nao_encontrados = 0

    for _, row in df.iterrows():
        questao_id = str(row['question'])
        usuario_id = str(row['usuario'])
        caminho = index.get((usuario_id, questao_id))
        if caminho is None:
            nao_encontrados += 1
            continue
        grupo = get_subpasta_nome(row['misconceptions_detectados'])
        grupos.setdefault(grupo, []).append({
            'question': questao_id,
            'usuario': usuario_id,
            'misconceptions_esperados': [] if grupo == 'SEM_MISCONCEPTIONS' else sorted([x for x in grupo.split('_') if x]),
            'path': caminho['path'] if isinstance(caminho, dict) else caminho,
            'assessment': caminho.get('assessment', '') if isinstance(caminho, dict) else '',
            'assessment_title': caminho.get('assessment_title', '') if isinstance(caminho, dict) else '',
            'assessment_type': caminho.get('assessment_type', '') if isinstance(caminho, dict) else '',
            'scope': caminho.get('scope', '') if isinstance(caminho, dict) else '',
            'tp_group': caminho.get('tp_group', 'Não classificado') if isinstance(caminho, dict) else 'Não classificado',
            'tp_description': caminho.get('tp_description', '') if isinstance(caminho, dict) else '',
        })
        encontrados += 1

    pasta_destino.mkdir(parents=True, exist_ok=True)
    references_output.write_text(json.dumps({
        'schema_version': 1,
        'description': 'Referências agrupadas por misconception; o código é extraído do último SUBMITION nos execution logs.',
        'source_references': str(references_path),
        'groups': grupos,
        'statistics': {'referencias': encontrados, 'nao_encontradas': nao_encontrados},
    }, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'Referências salvas em: {references_output}')
    print(f'Arquivos encontrados: {encontrados}; não encontrados: {nao_encontrados}')


criar_referencias_pc3()


In [ ]:
import json
from pathlib import Path
import sys
import os
from collections import Counter
import ast
import shutil
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
def localizar_raiz_projeto():
    # Import local para permitir execução isolada desta célula.
    from pathlib import Path as _Path
    for candidato in [_Path.cwd(), *_Path.cwd().parents]:
        if (candidato / 'Etapa_2' / 'output' / 'referencias_processamento.json').exists():
            return candidato
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto QuestionInsight.')

PROJECT_ROOT = localizar_raiz_projeto()
DETAIL_DIR = PROJECT_ROOT / 'Etapa_5' / 'Analise_Detalhada_Misconceptions_PC3'
OUTPUT_DIR = DETAIL_DIR / 'output'
sys.path.insert(0, str(PROJECT_ROOT / 'Etapa_5'))
# ============================================================
# CONFIGURAÇÕES
# ============================================================
# Caminhos resolvidos a partir da raiz do projeto, não do cwd do Jupyter.
references_output = OUTPUT_DIR / 'referencias_por_misconception.json'
pasta_raiz = str(OUTPUT_DIR)
MAX_WORKERS = 32
NUMERO_ANALISES = 1  # -1 = todos

# Constantes MC³ — EXATAMENTE as mesmas do código original
C4_MAX_ALLOWED_RANGEITER = 50
E2_MAX_ALLOWED_LISTS = 5
G4_MIN_VAR_CHRS = 4
G4_MIN_FNC_CHRS = 8
G4_MAX_ALLOWED_NONSIGNIFICANT = 70

DESCRICOES_MC = {
    'A2': 'Variável atribuída a si mesma',
    'A3': 'Variável inicializada desnecessariamente',
    'A4': 'Redefinição de built-in',
    'A5': 'Importação não utilizada',
    'B4': 'Comandos repetidos dentro de blocos if-elif-else',
    'B6': 'Comparação booleana tentada com loop while',
    'B8': 'Não utilização de elif/else',
    'B9': 'elif/else retestando condições já verificadas',
    'B10': 'elif/else desnecessário',
    'B11': 'Ifs distintos com blocos idênticos',
    'B12': 'Declarações if consecutivas iguais com operações distintas',
    'C1': 'Condição while testada novamente dentro do seu bloco',
    'C2': 'Loop redundante ou desnecessário',
    'C3': 'Operações redundantes dentro do loop',
    'C4': 'Número arbitrário de execuções de for loop ao invés de while',
    'C8': 'Loop for com sua variável de iteração sobrescrita',
    'D4': 'Variável fora do escopo da função',
    'E1': 'Verificação desnecessária de todas as combinações possíveis',
    'E2': 'Uso redundante ou desnecessário de listas',
    'G4': 'Funções/variáveis com nomes não significativos',
    'G5': 'Organização arbitrária de declarações',
    'H1': 'Declaração sem efeito'
}

def extrair_codigo_ultimo_submition(execution_path):
    """Extrai somente o código do último SUBMITION com GRADE e sem ERROR."""
    import re
    separador = '*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*'
    try:
        conteudo = Path(execution_path).read_text(encoding='utf-8', errors='replace')
    except (OSError, UnicodeError):
        return None
    blocos = [b for b in conteudo.split(separador)
              if re.search(r'^\s*==\s*SUBMITION\b', b, flags=re.I|re.M)]
    if not blocos:
        return None
    ultimo = blocos[-1]
    if (re.search(r'^\s*-+\s*GRADE\s*:', ultimo, flags=re.I|re.M) is None
            or re.search(r'^\s*-+\s*ERROR\s*:', ultimo, flags=re.I|re.M) is not None):
        return None
    match = re.search(r'^\s*--\s*CODE\s*:\s*\n(.*?)(?=^\s*--\s*(?:EXECUTION TIME|TEST CASE|ERROR|GRADE)\s*:|\Z)', ultimo, flags=re.I|re.M|re.S)
    if match is None:
        return None
    codigo = match.group(1).strip('\n\r')
    return codigo if codigo.strip() else None

# ============================================================
# MESMA FUNÇÃO analisar_codigo() DO CÓDIGO ORIGINAL
# ============================================================

def analisar_codigo(filepath, source_code=None):
    """Usa exatamente o mesmo agregado de detectores do parser principal."""
    try:
        if source_code is None:
            with open(filepath, 'r', encoding='utf-8') as file:
                code = file.read()
        else:
            code = source_code
        if not code or not code.strip():
            return [], ""
        parsed = ast.parse(code)
        results = VisitorMC3().analyze_all(parsed, MC3Config(
            c4_range_threshold=C4_MAX_ALLOWED_RANGEITER,
            e2_list_threshold=E2_MAX_ALLOWED_LISTS,
            g4_min_var_chars=G4_MIN_VAR_CHRS,
            g4_min_func_chars=G4_MIN_FNC_CHRS,
            g4_max_nonsignificant_percent=G4_MAX_ALLOWED_NONSIGNIFICANT,
        ))
        detectados = []
        for mc, resultado in results.items():
            valor = resultado[0] if isinstance(resultado, tuple) else resultado
            if valor:
                detectados.append(mc)
        return sorted(detectados), code
    except Exception:
        return [], ""
# ============================================================
# LOCALIZAR OCORRÊNCIAS DE CADA MC NO CÓDIGO
# ============================================================

def localizar_ocorrencias(parsed, code, mcs_detectados):
    resultados = {}
    linhas_codigo = code.splitlines()

    def nodes_iguais(n1, n2):
        if isinstance(n1, ast.Name) and isinstance(n2, ast.Name): return n1.id == n2.id
        if isinstance(n1, ast.Constant) and isinstance(n2, ast.Constant): return n1.value == n2.value
        return False

    def ops_inversas(ops1, ops2):
        inv = {ast.Eq: ast.NotEq, ast.NotEq: ast.Eq, ast.Lt: ast.GtE,
               ast.LtE: ast.Gt, ast.Gt: ast.LtE, ast.GtE: ast.Lt,
               ast.In: ast.NotIn, ast.NotIn: ast.In}
        if len(ops1) != 1 or len(ops2) != 1: return False
        return inv.get(type(ops1[0])) == type(ops2[0])

    def fallback(mc):
        return [{'linha': 0, 'descricao': f'{mc} detectado pelo VisitorMC3 (localização exata não mapeada)', 'trecho': '—'}]

    # ---------- A2 ----------
    if 'A2' in mcs_detectados:
        ocs = []
        for node in ast.walk(parsed):
            if isinstance(node, ast.Assign):
                for target in node.targets:
                    if isinstance(target, ast.Name) and isinstance(node.value, ast.Name):
                        if target.id == node.value.id:
                            ocs.append({'linha': node.lineno,
                                        'descricao': f"'{target.id}' atribuído a si mesmo",
                                        'trecho': ast.unparse(node)})
        resultados['A2'] = ocs or fallback('A2')

    # ---------- A3 ----------
    if 'A3' in mcs_detectados:
        ocs = []
        def checar_a3(scope_node, is_func=False):
            declared = {}
            used = set()
            skip = set()
            if is_func:
                for arg in scope_node.args.args: skip.add(arg.arg)
                walk = list(ast.walk(scope_node))
            else:
                walk = [c for n in ast.iter_child_nodes(scope_node)
                        if not isinstance(n, ast.FunctionDef)
                        for c in ast.walk(n)]
            for node in walk:
                if isinstance(node, ast.Assign):
                    for t in node.targets:
                        if isinstance(t, ast.Name) and t.id not in skip and t.id not in declared:
                            declared[t.id] = node.lineno
            for node in walk:
                if isinstance(node, ast.Name) and isinstance(node.ctx, ast.Load):
                    used.add(node.id)
            for var, linha in declared.items():
                if var not in used:
                    ocs.append({'linha': linha,
                                'descricao': f"Variável '{var}' declarada mas nunca utilizada",
                                'trecho': f"{var} = ..."})
        checar_a3(parsed, is_func=False)
        for node in ast.walk(parsed):
            if isinstance(node, ast.FunctionDef): checar_a3(node, is_func=True)
        seen = set()
        uniq = [o for o in ocs if o['linha'] not in seen and not seen.add(o['linha'])]
        resultados['A3'] = uniq or fallback('A3')

    # ---------- A4 ----------
    if 'A4' in mcs_detectados:
        builtins = {
            'abs','all','any','ascii','bin','bool','bytearray','bytes','callable','chr',
            'classmethod','compile','complex','delattr','dict','dir','divmod','enumerate',
            'eval','exec','filter','float','format','frozenset','getattr','hasattr','hash',
            'help','hex','id','input','int','isinstance','issubclass','iter','len','list',
            'locals','map','max','memoryview','min','next','object','oct','open','ord','pow',
            'print','property','range','repr','reversed','round','set','setattr','slice',
            'sorted','staticmethod','str','sum','super','tuple','type','vars','zip'
        }
        ocs = []
        for node in ast.walk(parsed):
            if isinstance(node, ast.Assign):
                for tgt in node.targets:
                    if isinstance(tgt, ast.Name) and tgt.id in builtins:
                        ocs.append({'linha': node.lineno,
                                    'descricao': f"Built-in '{tgt.id}' redefinido como variável",
                                    'trecho': ast.unparse(node)})
            elif isinstance(node, ast.FunctionDef):
                if node.name in builtins:
                    ocs.append({'linha': node.lineno,
                                'descricao': f"Built-in '{node.name}' redefinido como função",
                                'trecho': f"def {node.name}(...)"})
                for arg in node.args.args:
                    if arg.arg in builtins:
                        ocs.append({'linha': node.lineno,
                                    'descricao': f"Built-in '{arg.arg}' usado como parâmetro",
                                    'trecho': f"def {node.name}(..., {arg.arg}, ...)"})
        resultados['A4'] = ocs or fallback('A4')

    # ---------- A5 ----------
    if 'A5' in mcs_detectados:
        imports = {}
        used_names = set()
        for node in ast.walk(parsed):
            if isinstance(node, ast.Import):
                for alias in node.names:
                    nome = alias.asname or alias.name.split('.')[0]
                    imports[nome] = node.lineno
            elif isinstance(node, ast.ImportFrom):
                for alias in node.names:
                    if alias.name != '*':
                        nome = alias.asname or alias.name
                        imports[nome] = node.lineno
                    else:
                        mod = node.module or '?'
                        imports[f'* (de {mod})'] = node.lineno
            elif isinstance(node, ast.Name):
                used_names.add(node.id)
        ocs = [{'linha': linha,
                'descricao': f"'{nome}' importado mas nunca utilizado",
                'trecho': linhas_codigo[linha-1].strip() if linha <= len(linhas_codigo) else "?"}
               for nome, linha in imports.items() if nome not in used_names]
        resultados['A5'] = ocs or fallback('A5')

    # ---------- B4 ----------
    if 'B4' in mcs_detectados:
        ocs = []
        for node in ast.walk(parsed):
            if isinstance(node, ast.If):
                blocks = []
                current = node
                while isinstance(current, ast.If):
                    blocks.append((current.lineno, current.body))
                    if len(current.orelse) == 1 and isinstance(current.orelse[0], ast.If):
                        current = current.orelse[0]
                    else:
                        if current.orelse: blocks.append((current.lineno, current.orelse))
                        break
                seen_b = {}
                for linha, body in blocks:
                    rep = "|".join([ast.dump(s) for s in body])
                    if rep in seen_b:
                        ocs.append({'linha': linha,
                                    'descricao': f"Bloco idêntico ao da linha {seen_b[rep]}",
                                    'trecho': ast.unparse(body[0]) if body else "..."})
                    else:
                        seen_b[rep] = linha
        resultados['B4'] = ocs or fallback('B4')

    # ---------- B6 ----------
    if 'B6' in mcs_detectados:
        ocs = []
        for node in ast.walk(parsed):
            if isinstance(node, ast.While) and isinstance(node.test, (ast.Compare, ast.BoolOp)):
                for item in ast.walk(node):
                    if isinstance(item, ast.Break):
                        ocs.append({'linha': node.lineno,
                                    'descricao': "while com comparação booleana e break — use condição direta",
                                    'trecho': f"while {ast.unparse(node.test)}: ... break"})
                        break
        resultados['B6'] = ocs or fallback('B6')

    # ---------- B8 ----------
    if 'B8' in mcs_detectados:
        ocs = []
        for node in ast.walk(parsed):
            if isinstance(node, ast.If):
                if len(node.orelse) > 0 and isinstance(node.orelse[0], ast.If):
                    if len(node.orelse[0].orelse) == 0:
                        ocs.append({'linha': node.lineno,
                                    'descricao': "Cadeia if/elif sem bloco else final",
                                    'trecho': f"if ... (linha {node.lineno}) / elif ... sem else"})
        resultados['B8'] = ocs or fallback('B8')

    # ---------- B9 ----------
    if 'B9' in mcs_detectados:
        ocs = []
        for node in ast.walk(parsed):
            if isinstance(node, ast.If) and isinstance(node.test, ast.Compare) and node.orelse:
                for chd in node.orelse:
                    if isinstance(chd, ast.If) and isinstance(chd.test, ast.Compare):
                        if (nodes_iguais(node.test.left, chd.test.left) and
                            ops_inversas(node.test.ops, chd.test.ops) and
                            len(node.test.comparators)==1 and len(chd.test.comparators)==1 and
                            nodes_iguais(node.test.comparators[0], chd.test.comparators[0])):
                            ocs.append({'linha': chd.lineno,
                                        'descricao': f"elif retesta condição inversa do if da linha {node.lineno}",
                                        'trecho': f"if {ast.unparse(node.test)} ... elif {ast.unparse(chd.test)}"})
        resultados['B9'] = ocs or fallback('B9')

    # ---------- B10 ----------
    if 'B10' in mcs_detectados:
        ocs = []
        for node in ast.walk(parsed):
            if isinstance(node, ast.If):
                def vazio(body): return not body or (len(body)==1 and isinstance(body[0], ast.Pass))
                if vazio(node.body) and node.orelse:
                    ocs.append({'linha': node.lineno,
                                'descricao': "if com corpo vazio/pass seguido de else/elif desnecessário",
                                'trecho': linhas_codigo[node.lineno-1].strip()})
        resultados['B10'] = ocs or fallback('B10')

    # ---------- B11 ----------
    if 'B11' in mcs_detectados:
        ocs = []
        seen_b = {}
        for node in ast.iter_child_nodes(parsed):
            if isinstance(node, ast.If):
                rep = "|".join([ast.dump(s) for s in node.body])
                if rep in seen_b:
                    ocs.append({'linha': node.lineno,
                                'descricao': f"Bloco idêntico ao if da linha {seen_b[rep]}",
                                'trecho': ast.unparse(node.body[0]) if node.body else "..."})
                else:
                    seen_b[rep] = node.lineno
        resultados['B11'] = ocs or fallback('B11')

    # ---------- B12 ----------
    if 'B12' in mcs_detectados:
        ocs = []
        for node in ast.walk(parsed):
            prev_if = None
            for child in ast.iter_child_nodes(node):
                if isinstance(child, ast.If) and not child.orelse:
                    if prev_if is not None:
                        cond_igual = False
                        if isinstance(prev_if.test, ast.Name) and isinstance(child.test, ast.Name):
                            cond_igual = prev_if.test.id == child.test.id
                        elif isinstance(prev_if.test, ast.Compare) and isinstance(child.test, ast.Compare):
                            cond_igual = (nodes_iguais(prev_if.test.left, child.test.left) and
                                          len(prev_if.test.ops)==1 and len(child.test.ops)==1 and
                                          type(prev_if.test.ops[0])==type(child.test.ops[0]))
                        if cond_igual:
                            ocs.append({'linha': child.lineno,
                                        'descricao': f"If consecutivo com mesma condição do if da linha {prev_if.lineno}",
                                        'trecho': f"if {ast.unparse(child.test)} ..."})
                    prev_if = child
                else:
                    prev_if = None
        resultados['B12'] = ocs or fallback('B12')

    # ---------- C1 ----------
    if 'C1' in mcs_detectados:
        ocs = []
        for node in ast.walk(parsed):
            if isinstance(node, ast.While) and isinstance(node.test, ast.Compare):
                for item in node.body:
                    if isinstance(item, ast.If) and isinstance(item.test, ast.Compare):
                        if nodes_iguais(node.test.left, item.test.left) and ops_inversas(node.test.ops, item.test.ops):
                            ocs.append({'linha': item.lineno,
                                        'descricao': f"If retesta condição inversa do while da linha {node.lineno}",
                                        'trecho': f"while {ast.unparse(node.test)} → if {ast.unparse(item.test)}"})
        resultados['C1'] = ocs or fallback('C1')

    # ---------- C2 ----------
    if 'C2' in mcs_detectados:
        ocs = []
        for node in ast.walk(parsed):
            if isinstance(node, ast.While):
                if isinstance(node.test, ast.Constant) and node.test.value is True:
                    for item in node.body:
                        if isinstance(item, ast.Break):
                            ocs.append({'linha': node.lineno,
                                        'descricao': "while True com break imediato — loop redundante",
                                        'trecho': "while True: ... break"})
            if isinstance(node, ast.For):
                if isinstance(node.iter, ast.Call):
                    if isinstance(node.iter.func, ast.Name) and node.iter.func.id == "range":
                        if len(node.iter.args)==1 and isinstance(node.iter.args[0], ast.Constant):
                            if node.iter.args[0].value == 1:
                                ocs.append({'linha': node.lineno,
                                            'descricao': "for range(1) executa apenas 1 vez",
                                            'trecho': ast.unparse(node.iter)})
        resultados['C2'] = ocs or fallback('C2')

    # ---------- C3 ----------
    if 'C3' in mcs_detectados:
        ocs = []
        for node in ast.walk(parsed):
            if isinstance(node, (ast.For, ast.While)):
                seen_s = {}
                for stmt in node.body:
                    rep = ast.dump(stmt)
                    if rep in seen_s:
                        ocs.append({'linha': stmt.lineno,
                                    'descricao': f"Operação idêntica à linha {seen_s[rep]} repetida no loop",
                                    'trecho': ast.unparse(stmt)})
                    else:
                        seen_s[rep] = stmt.lineno
        resultados['C3'] = ocs or fallback('C3')

    # ---------- C4 ----------
    if 'C4' in mcs_detectados:
        ocs = []
        for node in ast.walk(parsed):
            if isinstance(node, ast.For):
                if isinstance(node.iter, ast.Call):
                    if isinstance(node.iter.func, ast.Name) and node.iter.func.id == "range":
                        if len(node.iter.args)==1 and isinstance(node.iter.args[0], ast.Constant):
                            val = node.iter.args[0].value
                            if val >= C4_MAX_ALLOWED_RANGEITER:
                                ocs.append({'linha': node.lineno,
                                            'descricao': f"for range({val}) com valor fixo alto — considere while",
                                            'trecho': ast.unparse(node.iter)})
        resultados['C4'] = ocs or fallback('C4')

    # ---------- C8 ----------
    if 'C8' in mcs_detectados:
        ocs = []
        def checar_c8(for_node, externas=None):
            if externas is None: externas = []
            iter_vars = []
            if isinstance(for_node.target, ast.Name): iter_vars.append(for_node.target.id)
            elif isinstance(for_node.target, (ast.Tuple, ast.List)):
                for e in for_node.target.elts:
                    if isinstance(e, ast.Name): iter_vars.append(e.id)
            todas = externas + iter_vars
            for stmt in for_node.body:
                if isinstance(stmt, ast.Assign):
                    for tgt in stmt.targets:
                        if isinstance(tgt, ast.Name) and tgt.id in todas:
                            ocs.append({'linha': stmt.lineno,
                                        'descricao': f"Variável de iteração '{tgt.id}' sobrescrita no loop",
                                        'trecho': ast.unparse(stmt)})
                elif isinstance(stmt, ast.AugAssign):
                    if isinstance(stmt.target, ast.Name) and stmt.target.id in todas:
                        ocs.append({'linha': stmt.lineno,
                                    'descricao': f"Variável de iteração '{stmt.target.id}' modificada no loop",
                                    'trecho': ast.unparse(stmt)})
                elif isinstance(stmt, ast.For):
                    checar_c8(stmt, todas)
        for node in ast.walk(parsed):
            if isinstance(node, ast.For): checar_c8(node)
        resultados['C8'] = ocs or fallback('C8')

    # ---------- D4 ----------
    if 'D4' in mcs_detectados:
        ocs = []
        global_vars = set()
        for node in ast.iter_child_nodes(parsed):
            if not isinstance(node, ast.FunctionDef):
                for chd in ast.walk(node):
                    if isinstance(chd, ast.Assign):
                        for tgt in chd.targets:
                            if isinstance(tgt, ast.Name): global_vars.add(tgt.id)
        for node in ast.walk(parsed):
            if isinstance(node, ast.FunctionDef):
                local_vars = set(arg.arg for arg in node.args.args)
                for stmt in node.body:
                    if isinstance(stmt, ast.Assign):
                        for tgt in stmt.targets:
                            if isinstance(tgt, ast.Name): local_vars.add(tgt.id)
                for chd in ast.walk(node):
                    if isinstance(chd, ast.Name) and isinstance(chd.ctx, ast.Load):
                        if chd.id in global_vars and chd.id not in local_vars:
                            ocs.append({'linha': chd.lineno,
                                        'descricao': f"Global '{chd.id}' usada em '{node.name}' sem ser parâmetro",
                                        'trecho': chd.id})
        if ocs:
            seen = set()
            ocs = [o for o in ocs if (o['linha'], o['descricao']) not in seen
                   and not seen.add((o['linha'], o['descricao']))]
        resultados['D4'] = ocs or fallback('D4')

    # ---------- E1 ----------
    if 'E1' in mcs_detectados:
        ocs = []
        for node in ast.walk(parsed):
            if isinstance(node, ast.If) and isinstance(node.test, ast.BoolOp):
                if len(node.test.values) > 5:
                    ocs.append({'linha': node.lineno,
                                'descricao': f"{len(node.test.values)} condições em um único if — verificação excessiva",
                                'trecho': ast.unparse(node.test)})
        resultados['E1'] = ocs or fallback('E1')

    # ---------- E2 ----------
    if 'E2' in mcs_detectados:
        listas = []
        for node in ast.walk(parsed):
            if isinstance(node, ast.Assign):
                if isinstance(node.value, (ast.List, ast.ListComp)):
                    listas.append({'linha': node.lineno, 'trecho': ast.unparse(node)})
        ocs = [{'linha': l['linha'],
                'descricao': f"Lista declarada (total: {len(listas)}) — uso excessivo",
                'trecho': l['trecho']} for l in listas]
        resultados['E2'] = ocs or fallback('E2')

    # ---------- G4 ----------
    if 'G4' in mcs_detectados:
        ocs = []
        var_names = []
        func_names = []
        for node in ast.walk(parsed):
            if isinstance(node, ast.Assign):
                for tgt in node.targets:
                    if isinstance(tgt, ast.Name): var_names.append((tgt.id, node.lineno))
            elif isinstance(node, ast.FunctionDef):
                func_names.append((node.name, node.lineno))
        curtos_vars = [(n, l) for n, l in var_names if len(n) <= G4_MIN_VAR_CHRS]
        curtos_funcs = [(n, l) for n, l in func_names if len(n) <= G4_MIN_FNC_CHRS]
        if len(var_names) > 0 and len(curtos_vars)/len(var_names) >= G4_MAX_ALLOWED_NONSIGNIFICANT/100:
            for nome, linha in curtos_vars:
                ocs.append({'linha': linha,
                            'descricao': f"Variável '{nome}' com nome muito curto (≤{G4_MIN_VAR_CHRS} chars)",
                            'trecho': f"{nome} = ..."})
        if len(func_names) > 0 and len(curtos_funcs)/len(func_names) >= G4_MAX_ALLOWED_NONSIGNIFICANT/100:
            for nome, linha in curtos_funcs:
                ocs.append({'linha': linha,
                            'descricao': f"Função '{nome}' com nome muito curto (≤{G4_MIN_FNC_CHRS} chars)",
                            'trecho': f"def {nome}(...)"})
        resultados['G4'] = ocs or fallback('G4')

    # ---------- G5 ----------
    if 'G5' in mcs_detectados:
        ocs = []
        seen_func = False
        seen_code_after = False
        for node in ast.iter_child_nodes(parsed):
            if isinstance(node, (ast.Import, ast.ImportFrom)): continue
            if isinstance(node, ast.Expr) and isinstance(node.value, ast.Constant): continue
            if isinstance(node, ast.FunctionDef):
                if seen_code_after:
                    ocs.append({'linha': node.lineno,
                                'descricao': f"Função '{node.name}' declarada após código executável",
                                'trecho': f"def {node.name}(...)"})
                seen_func = True
            else:
                if seen_func: seen_code_after = True
        resultados['G5'] = ocs or fallback('G5')

    # ---------- H1 ----------
    if 'H1' in mcs_detectados:
        ocs = []
        for node in ast.walk(parsed):
            if isinstance(node, ast.Expr):
                if isinstance(node.value, ast.Constant):
                    if not isinstance(node.value.value, str):
                        ocs.append({'linha': node.lineno,
                                    'descricao': f"Expressão '{node.value.value}' não tem efeito",
                                    'trecho': str(node.value.value)})
        resultados['H1'] = ocs or fallback('H1')

    return resultados


# ============================================================
# PIPELINE PRINCIPAL
# ============================================================

def detectar_misconceptions_detalhado(filepath):
    """Analisa o código extraído do último SUBMITION do execution log."""
    code_fonte = extrair_codigo_ultimo_submition(filepath)
    if code_fonte is None:
        return None, "Execution sem último SUBMITION válido"
    mcs_detectados, code = analisar_codigo(filepath, source_code=code_fonte)
    if not code:
        return None, "Arquivo vazio ou erro de parse"

    try:
        parsed = ast.parse(code)
    except Exception as e:
        return None, f"Erro ao parsear: {e}"

    resultados = localizar_ocorrencias(parsed, code, mcs_detectados)
    return resultados, code


# ============================================================
# GERAÇÃO DO RELATÓRIO .TXT
# ============================================================

def gerar_txt(filepath_py, resultados, code, metadata=None):
    nome_arquivo = os.path.basename(filepath_py)
    metadata = metadata or {}
    partes = os.path.splitext(nome_arquivo)[0].rsplit('_', 1)
    assessment_id = str(metadata.get('assessment') or (partes[0] if len(partes) == 2 else '?'))
    questao_id = str(metadata.get('question') or (partes[1] if len(partes) == 2 else '?'))
    usuario_id = str(metadata.get('usuario') or '?')
    tp_group = metadata.get('tp_group', 'Não classificado')
    tp_description = metadata.get('tp_description', 'Assessment sem grupo TP identificado')
    assessment_title = metadata.get('assessment_title', '')
    linhas_codigo = code.splitlines()

    linhas = []
    linhas.append("=" * 70)
    linhas.append("RELATÓRIO DE ANÁLISE DE MISCONCEPTIONS (MC³)")
    linhas.append("=" * 70)
    linhas.append(f"Arquivo     : {nome_arquivo}")
    linhas.append(f"Caminho     : {os.path.abspath(filepath_py)}")
    linhas.append(f"Assessment  : {assessment_id}")
    if assessment_title:
        linhas.append(f"Título      : {assessment_title}")
    linhas.append(f"Questão     : {questao_id}")
    linhas.append(f"Aluno       : {usuario_id}")
    linhas.append(f"Grupo       : {tp_group}")
    linhas.append(f"Significado : {tp_description}")
    linhas.append(f"Total MCs   : {len(resultados)}")
    linhas.append("")

    if not resultados:
        linhas.append("✅ Nenhum misconception detectado neste código.")
    else:
        linhas.append(f"⚠️  Misconceptions detectados: {', '.join(sorted(resultados.keys()))}")
        linhas.append("")
        linhas.append("-" * 70)
        linhas.append("CÓDIGO FONTE")
        linhas.append("-" * 70)
        for i, linha in enumerate(linhas_codigo, 1):
            linhas.append(f"  {i:4d} | {linha}")
        linhas.append("")
        linhas.append("-" * 70)
        linhas.append("DETALHAMENTO DOS MISCONCEPTIONS")
        linhas.append("-" * 70)

        for mc in sorted(resultados.keys()):
            ocorrencias = resultados[mc]
            desc_geral = DESCRICOES_MC.get(mc, mc)
            linhas.append("")
            linhas.append(f"▶ {mc} — {desc_geral}")
            linhas.append(f"  Ocorrências: {len(ocorrencias)}")
            for i, oc in enumerate(ocorrencias, 1):
                linhas.append("")
                linhas.append(f"  [{i}] Linha {oc['linha']}: {oc['descricao']}")
                linhas.append(f"      Trecho  : {oc['trecho']}")
                ln = oc['linha']
                if ln > 0:
                    inicio = max(0, ln - 2)
                    fim = min(len(linhas_codigo), ln + 1)
                    linhas.append(f"      Contexto:")
                    for idx in range(inicio, fim):
                        marker = ">>>" if idx == ln - 1 else "   "
                        linhas.append(f"        {marker} {idx+1:4d} | {linhas_codigo[idx]}")

    linhas.append("")
    linhas.append("=" * 70)
    return "\n".join(linhas)


# ============================================================
# PROCESSAMENTO DE UM ÚNICO ARQUIVO
# ============================================================

def processar_arquivo(args):
    subpasta_mc, filepath_py, metadata = args
    nome_base = os.path.splitext(os.path.basename(filepath_py))[0]
    nome_pasta_mc = os.path.basename(subpasta_mc)
    usuario_id = str(metadata.get('usuario', 'desconhecido'))
    subpasta_arquivo = os.path.join(subpasta_mc, f'{nome_base}__usuario_{usuario_id}')
    os.makedirs(subpasta_arquivo, exist_ok=True)
    esperados = metadata.get('misconceptions_esperados', [])
    resultados, code = detectar_misconceptions_detalhado(filepath_py)
    detectados = sorted(resultados.keys()) if resultados is not None else []
    erro_msg = None
    if resultados is None:
        erro_msg = code
        txt_content = (f"ERRO: Não foi possível analisar o arquivo.\n"
                       f"Caminho completo: {os.path.abspath(filepath_py)}\n"
                       f"Motivo: {code}")
        resultados = {}
        code = ""
    else:
        txt_content = gerar_txt(filepath_py, resultados, code, metadata)
    caminho_txt = os.path.join(subpasta_arquivo, nome_base + '.txt')
    with open(caminho_txt, 'w', encoding='utf-8') as f:
        f.write(txt_content)
    registro = {
        'path': os.path.abspath(filepath_py), 'arquivo': nome_base + '.log',
        'usuario': metadata.get('usuario', ''), 'question': metadata.get('question', ''),
        'grupo_pasta': nome_pasta_mc, 'misconceptions_esperados': esperados,
        'misconceptions_detectados': detectados, 'quantidade_esperada': len(esperados),
        'quantidade_detectada': len(detectados),
        'status': 'erro_parse' if not code else ('divergente' if esperados != detectados else 'ok'),
        'erro': erro_msg,
    }
    return nome_base, nome_pasta_mc, len(resultados), registro
# ============================================================
# EXECUÇÃO PRINCIPAL
# ============================================================

def analisar_codigos_separados(numero_analises=-1):
    # A célula 0 normalmente cria este arquivo. Se ela já foi carregada,
    # regenere automaticamente quando a saída tiver sido removida/limpa.
    if not references_output.exists():
        criar_referencias = globals().get('criar_referencias_pc3')
        if callable(criar_referencias):
            criar_referencias()
    if not references_output.exists():
        raise FileNotFoundError(
            f'Referências não encontradas em: {references_output.resolve()}. '
            'Execute a primeira célula do notebook para gerar '
            'referencias_por_misconception.json.'
        )
    referencias = json.loads(references_output.read_text(encoding='utf-8'))
    todos_paths = [item.get('path', '') for grupo in referencias.get('groups', {}).values() for item in grupo]
    paths_obsoletos = [path for path in todos_paths if str(path).lower().endswith('.py')]
    if paths_obsoletos:
        raise RuntimeError(
            'referencias_por_misconception.json está desatualizado: contém codes/*.py. '
            'Execute primeiro a célula 0 para regenerar referências com executions/*.log.'
        )
    tarefas = []
    for grupo, items in sorted(referencias.get('groups', {}).items()):
        pasta_grupo = os.path.join(pasta_raiz, grupo)
        os.makedirs(pasta_grupo, exist_ok=True)
        for item in items:
            caminho = item.get('path')
            if caminho and os.path.isfile(caminho):
                tarefas.append((pasta_grupo, caminho, item))
    if numero_analises != -1:
        tarefas = tarefas[:numero_analises]
    total = len(tarefas)
    print(f'Total de arquivos referenciados: {total:,}')
    print(f'Workers: {MAX_WORKERS}')
    print('=' * 70)
    processados = 0
    erros = 0
    resultados_execucao = []
    pastas_criadas = set()
    contagem_por_pasta = Counter()
    print_lock = threading.Lock()
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(processar_arquivo, tarefa): tarefa for tarefa in tarefas}
        with tqdm(total=total, desc='Analisando executions', unit='arquivo', dynamic_ncols=True) as barra:
            for future in as_completed(futures):
                try:
                    nome_base, nome_pasta_mc, n_mcs, registro_resultado = future.result()
                    resultados_execucao.append(registro_resultado)
                    pastas_criadas.add(nome_pasta_mc)
                    contagem_por_pasta[nome_pasta_mc] += 1
                    processados += 1
                except Exception as error:
                    erros += 1
                    with print_lock:
                        # O erro continua sendo contado, mas não polui a saída de cada arquivo.
                        print(f'\nERRO durante o processamento: {error}')
                finally:
                    barra.update(1)
    resultados_path = Path(pasta_raiz) / 'analise_detalhada_resultados.json'
    resultados_path.write_text(json.dumps({
        'schema_version': 1,
        'descricao': 'Comparação entre misconceptions esperados no CSV e detectados no código do último SUBMITION.',
        'total_processados': len(resultados_execucao),
        'total_divergencias': sum(r['status'] == 'divergente' for r in resultados_execucao),
        'total_erros_parse': sum(r['status'] == 'erro_parse' for r in resultados_execucao),
        'contagem_esperada': dict(Counter(mc for r in resultados_execucao for mc in r['misconceptions_esperados'])),
        'contagem_detectada': dict(Counter(mc for r in resultados_execucao for mc in r['misconceptions_detectados'])),
        'resultados': sorted(resultados_execucao, key=lambda r: (str(r['usuario']), str(r['question']), r['path']))
    }, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print('\n' + '=' * 70)
    print('ANÁLISE CONCLUÍDA')
    print(f'  Processados: {processados:,}')
    print(f'  Erros: {erros:,}')
    print(f'  Resultados detalhados: {resultados_path.resolve()}')
    print('\nPASTAS DE MISCONCEPTIONS (PCs) CRIADAS')
    print(f'  Total de pastas: {len(pastas_criadas):,}')
    for pasta in sorted(pastas_criadas):
        print(f'  - {pasta}: {contagem_por_pasta[pasta]:,} arquivos')
    print('=' * 70)
analisar_codigos_separados(numero_analises=-1)
